In [2]:
import osmnx as ox
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import nearest_points
import pyproj
from google.cloud import storage
from math import radians, sin, cos, sqrt, atan2


## Defining Functions

In [3]:
def haversine(lon1, lat1, lon2, lat2):
    # Convert degrees to radians
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    
    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    distance = 6371 * c * 1000  # Earth radius in km, convert to meters
    return distance

def meters_to_degrees(meters, latitude):
    return meters / (111320 * cos(radians(latitude)))


## Extract Data using OSMNX
- Use OSMNX to get POI's within a certain radius of Austin city center 

In [4]:
# Define the latitude and longitude of the center point
latitude = 30.2645322996953
longitude = -97.7497791359203

# Define the radius in meters (25 km = 25000 meters)
radius = 35000

# Define amenities of interest
amenities_of_interest = {
    # food
    'cafe': ['coffee', 'green'],
    'bar': ['beer', 'green'],
    'restaurant': ['cutlery', 'green'],
    'pub': ['beer', 'green'],

    # social buildings
    'social_centre': ['institution', 'orange'],
    'library': ['book', 'orange'],
    'marketplace': ['shopping-cart', 'green'],
    'events_venue': ['institution', 'orange'],
    'exhibition_centre': ['institution', 'lightgrey'],
    'place_of_worship': ['institution', 'orange'],

    # places to chill
    'park': ['tree', 'green'],
    'music_venue': ['music', 'red'],
    'cinema': ['film', 'red'],
    'theatre': ['film', 'red']
}

# Define amenities of interest
amenities = list(amenities_of_interest.keys())
amenity_tags = {'amenity': amenities}

# Query the POIs within the specified radius
pois = ox.features.features_from_point((latitude, longitude), tags=amenity_tags, dist=radius)

# Filter the POIs to include only the amenities of interest
filtered_pois = pois[pois['amenity'].isin(amenities)]

# Create a new DataFrame with selected columns
columns_to_keep = ['amenity', 'name', 'geometry']  # Add other columns if needed
filtered_pois_df = filtered_pois[columns_to_keep]

# Add a column for the icon and color based on the amenities_of_interest dictionary
filtered_pois_df['icon'] = filtered_pois_df['amenity'].apply(lambda x: amenities_of_interest[x][0])
filtered_pois_df['color'] = filtered_pois_df['amenity'].apply(lambda x: amenities_of_interest[x][1])


c:\Users\Martin\anaconda3\envs\onebus\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
c:\Users\Martin\anaconda3\envs\onebus\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


#### Load Bus Stop Data

In [5]:
unique_bus_stops = pd.read_csv("all_unique_stops.csv")
unique_bus_stops

,Unnamed: 0,trip_id,arrival_time,departure_time,stop_sequence,trip_headsign,direction_id,route_id,wheelchair_accessible,bikes_allowed,stop_lat,stop_lon,stop_name,is_express
0,144455,2961157_3041,03:43:00,03:43:00,1,20 ABIA Airport SB,1,20,1,1,30.313865,-97.664697,7000 Manor/Susquehanna,False
1,144456,2961157_3041,03:44:13,03:44:13,2,20 ABIA Airport SB,1,20,1,1,30.310003,-97.667482,6650 Manor/Loyola,False
2,144457,2961157_3041,03:45:12,03:45:12,3,20 ABIA Airport SB,1,20,1,1,30.308500,-97.671213,University Hills SB (Manor/Northeast),False
3,144458,2961157_3041,03:46:50,03:46:50,4,20 ABIA Airport SB,1,20,1,1,30.307941,-97.678194,6102 Manor/Wheless,False
4,144459,2961157_3041,03:47:27,03:47:27,5,20 ABIA Airport SB,1,20,1,1,30.306590,-97.680101,5900 Manor/Sweeney,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3609,536954,2967774_13062,25:24:19,25:24:19,33,485 Downtown SB,1,485,1,1,30.262602,-97.721983,7th/Chicon,False
3610,536955,2967774_13062,25:25:38,25:25:38,34,485 Downtown SB,1,485,1,1,30.263880,-97.725646,7th/Concho,False
3611,536956,2967774_13062,25:27:38,25:27:38,35,485 Downtown SB,1,485,1,1,30.265561,-97.730500,1114 7th/Waller,False
3612,536957,2967774_13062,25:31:18,25:31:18,36,485 Downtown SB,1,485,1,1,30.268997,-97.738479,411 8th/Trinity,False


In [6]:
# Remove rows where the 'name' column is NaN
filtered_pois_df = filtered_pois_df.dropna(subset=['name','amenity'])

'''
General Logic:
1. Convert both bus stop and POI dataframe to geo dataframes
2. Bus Stop dataframe is converted to a "buffer" dataframe which is a polygon of N buffer distance given an x,y coordinate
3. Perform a spatial join between the POI and the bus stop buffer dataframe in order to get the POI accessible by bus stops

'''

# Step 1: Add a geometry column to the bus stop DataFrame
unique_bus_stops['geometry'] = unique_bus_stops.apply(lambda row: Point(row['stop_lon'], row['stop_lat']), axis=1)

# Step 2: Convert both DataFrames to GeoDataFrames
# Set the CRS to WGS84 (EPSG:4326) for latitude/longitude
bus_stops_gdf = gpd.GeoDataFrame(unique_bus_stops, geometry='geometry', crs='EPSG:4326')
pois_gdf = gpd.GeoDataFrame(filtered_pois_df, geometry='geometry', crs='EPSG:4326')

# Separate points and polygons
points_gdf = pois_gdf[pois_gdf.geometry.type == 'Point']
polygons_gdf = pois_gdf[pois_gdf.geometry.type == 'Polygon']

# Step 3: Create a copy for buffered bus stops
bus_stops_buffered_gdf = bus_stops_gdf.copy()

# Step 4: Calculate buffer distance in degrees

buffer_distance_degrees = meters_to_degrees(500, 30.26562)  # 500 meters at latitude 30.26562

# Step 5: Apply buffer to create circular areas around bus stops
bus_stops_buffered_gdf['geometry'] = bus_stops_buffered_gdf.geometry.buffer(buffer_distance_degrees)

# Step 6: Perform a spatial join to find POIs within the buffered area
pois_within_500m_points = gpd.sjoin(points_gdf,bus_stops_buffered_gdf,how='inner',predicate='within')
pois_within_500m_poly = gpd.sjoin(polygons_gdf, bus_stops_buffered_gdf, how='inner', predicate='intersects')

# Step 7: Calculate distance from centroid for polygons
pois_within_500m_poly['geometry'] = pois_within_500m_poly['geometry'].centroid

combined_poi = pd.concat([pois_within_500m_poly, pois_within_500m_points], ignore_index=True)


# Extract coordinates from geometry and stop_lat/stop_lon
combined_poi['distance'] = combined_poi.apply(
    lambda row: haversine(
        row.geometry.x, row.geometry.y,  # POI coordinates (lon, lat)
        row['stop_lon'], row['stop_lat']  # Bus stop coordinates
    ),
    axis=1
)

# Sort by distance (ascending) to prioritize shortest distances
combined_poi = combined_poi.sort_values('distance')

# Drop duplicates, keeping the closest POI per geometry
combined_poi_no_duplicates = combined_poi.drop_duplicates(subset=['geometry'], keep='first')

amenity_mapping = {
    'restaurant': 'Food Services',
    'marketplace': 'Commercial',
    'bar': 'Food Services',
    'cafe': 'Food Services',
    'place_of_worship': 'Community Services',
    'social_centre': 'Community Services',
    'events_venue': 'Recreation',
    'pub': 'Food Services',
    'library': 'Community Services',
    'cinema': 'Recreation',
    'theatre': 'Recreation',
    'music_venue': 'Recreation'
}

combined_poi_no_duplicates['amenity_category'] = combined_poi_no_duplicates['amenity'].map(amenity_mapping)

C:\Users\Martin\AppData\Local\Temp\ipykernel_25808\4189544917.py:32: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  bus_stops_buffered_gdf['geometry'] = bus_stops_buffered_gdf.geometry.buffer(buffer_distance_degrees)
C:\Users\Martin\AppData\Local\Temp\ipykernel_25808\4189544917.py:39: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois_within_500m_poly['geometry'] = pois_within_500m_poly['geometry'].centroid
c:\Users\Martin\anaconda3\envs\onebus\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-do

In [7]:
combined_poi_no_duplicates = combined_poi_no_duplicates[['amenity','amenity_category', 'name', 'geometry', 'icon', 'color']]

In [8]:
combined_poi_no_duplicates['amenity'].unique()

array(['restaurant', 'bar', 'marketplace', 'place_of_worship',
       'social_centre', 'events_venue', 'cafe', 'pub', 'library',
       'cinema', 'theatre', 'music_venue'], dtype=object)

In [9]:
combined_poi_no_duplicates

,amenity,amenity_category,name,geometry,icon,color
15836,restaurant,Food Services,Super Burrito,POINT (-97.75271 30.2695),cutlery,green
32578,restaurant,Food Services,Fig Italian Kitchen and Bar,POINT (-97.78598 30.2407),cutlery,green
32133,bar,Food Services,Green Light Social,POINT (-97.75039 30.27036),beer,green
41160,marketplace,Commercial,Sustainable Food Center Farmers' Market,POINT (-97.74711 30.26725),shopping-cart,green
30579,bar,Food Services,Kung Fu Saloon,POINT (-97.75037 30.27034),beer,green
...,...,...,...,...,...,...
27212,restaurant,Food Services,A+A Sichuan China,POINT (-97.78908 30.44444),cutlery,green
374,restaurant,Food Services,Cover 2,POINT (-97.79139 30.45719),cutlery,green
20875,restaurant,Food Services,Salata,POINT (-97.7329 30.3888),cutlery,green
39085,restaurant,Food Services,Fortune Garden,POINT (-97.6869 30.55628),cutlery,green


## Data Loading to GCS

In [14]:
combined_poi_no_duplicates.to_csv('filtered_pois.csv', index=False)
